In [1]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [2]:
EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
G0_EMBEDDINGS_FILE = CWD / "g0_embeddings.json"
DOCUMENT_CHUNK_STORE = CWD / "result.json"
MAX_TOP_CHUNKS = 5

USER_PROMPT = "What architectural modification did the authors make to the GRU model from BotScreen?"

In [3]:
# load g0 entity embeddings from file
import json
from utils.models import EntityDescEmbed
entity_embeddings:list[EntityDescEmbed] = []
with open(G0_EMBEDDINGS_FILE, "r") as ef:
    entity_embeddings = json.load(ef)

/home/nathan/Projects/X-RAG/src/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import litellm

resp = await litellm.aembedding(model=EMBED_MODEL, input=USER_PROMPT)
data = resp['data'][0]
prompt_embedding:list[float] = data['embedding']

In [5]:
import numpy as np

# dense vector search function
def search_dense(query_vec: list[float], entity_store: list[dict], topk: int= 10):
    if topk <= 0:
        return []

    # Convert to NumPy arrays
    query = np.asarray(query_vec, dtype=np.float32)
    entity_matrix = np.asarray(
        [item["desc_embed"] for item in entity_store], dtype=np.float32
    )

    # Normalize to use cosine similarity (handle zero vectors defensively)
    q_norm = np.linalg.norm(query)
    if q_norm == 0:
        raise ValueError("query vector has zero norm")
    query /= q_norm

    e_norms = np.linalg.norm(entity_matrix, axis=1, keepdims=True)
    e_norms[e_norms == 0] = 1.0
    entity_matrix = entity_matrix / e_norms

    # Dot product gives cosine similarity
    sims = entity_matrix @ query

    # Grab top-k indices
    k = min(topk, len(entity_store))
    top_idx = np.argpartition(-sims, k - 1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    return [
        {**entity_store[i], "score": float(sims[i])}
        for i in top_idx
    ]

seed_entities = search_dense(prompt_embedding, entity_embeddings)

In [6]:
# assemble ancestor chains
from utils import mg_driver
await mg_driver.init()

ancestor_chains = []
for s in seed_entities:
    chain = await mg_driver.get_ancestor_chain(s['key'])
    ancestor_chains.append(chain)

In [7]:
# def pprint_anc_chain(ancestor_chain):
#     s=""
#     for i,e in enumerate(ancestor_chain):
#         if 'layer' in e.keys():
#             s += f"{e['layer']}: "
#         else: s+= "?: "

#         s+= e['key']

#         if i != len(ancestor_chain)-1:
#             s+= " -> "
#     print(s)

In [8]:
# form lca paths from ancestor chains
from itertools import combinations

reasoning_path_information:dict[str,dict] = {}
lca_encountered_agg_entities:dict[str, dict] = {}
entity_parent_map:dict[str,str] = {}
entity_info_map:dict[str,dict] = {}

for s1, s2 in combinations(ancestor_chains, 2):
    zipped_chain = zip(s1,s2)
    lca_key:str = None
    for anc1, anc2 in zipped_chain:
        if anc1['key'] == anc2['key']:
            lca_key = anc1['key']
            break
    if not lca_key:
        raise RuntimeError(f"could not find lca for '{s1[0]['key']}' TO '{s2[0]['key']}'")
    # print(f"LCA: '{s1[0]['key']}' TO '{s2[0]['key']}' = '{lca_key}'")

    # find path from s1 -> lca -> s2
    lca_path = await mg_driver.get_lca_path(s1[0]['key'], s2[0]['key'], lca_key)
    
    # LCA PATH GOES entity_1 -> agg_1 -> ... -> root/aggX -> ... -> agg_1 -> entity_2
    # should always be at least 3 nodes long
    if lca_path[0]['key'] not in entity_parent_map.keys():
        entity_parent_map[lca_path[0]['key']] = lca_path[1] # parent of entity 1
        entity_info_map[lca_path[0]['key']] = lca_path[0]
    if lca_path[-1]['key'] not in entity_parent_map.keys():
        entity_parent_map[lca_path[-1]['key']] = lca_path[-2] # parent of entity 2
        entity_info_map[lca_path[-1]['key']] = lca_path[-1]

    for i, ent in enumerate(lca_path):
        # populate the encountered aggregate entities dict
        if ent['key'] == 'root': # only want to collect seen aggregate nodes , ignore root
            continue
        if ent['key'] in lca_encountered_agg_entities.keys():
            continue
        lca_encountered_agg_entities[ent['key']] = ent

    # collect horizontal intra-relations between nodes in the path
    connection_edges = await mg_driver.get_intra_lca_path_links(s1[0]['key'], s2[0]['key'], lca_key)
    if len(connection_edges) > 0:
        for ce in connection_edges:
            if ce['r_key'] in reasoning_path_information.keys():
                continue
            reasoning_path_information[ce['r_key']] = ce

In [9]:
# chunk text from the top K seed entities

top_chunks = await mg_driver.get_ranked_provenance_for_entitys([e['key'] for e in seed_entities])
top_chunks = set([c['chunk_id'] for c in top_chunks[:MAX_TOP_CHUNKS]])

In [10]:
# collect raw chunk text for top k chunks:
top_chunks_text = []
import ijson
with open(DOCUMENT_CHUNK_STORE, "rb") as in_file:
    for itm in ijson.items(in_file, "item"):
        for chunk in itm['chunks']:
            if chunk['id'] in top_chunks:
                top_chunks_text.append(chunk['raw_text'])

In [11]:
#  ask LLM for response with our retrieved data
from utils.signatures import generate_augmented_response

context_ent_info = [(entity_info_map[e]['name'], entity_parent_map[e]['name'], entity_info_map[e]['desc']) for e in entity_info_map]
context_rpath_info = [ vv['r_desc'] for vv in [v for _,v in reasoning_path_information.items()]]
context_agg_ent_info = [(vv['name'], vv['desc']) for vv in [v for v in lca_encountered_agg_entities.values()]]

response = await generate_augmented_response(
    query = USER_PROMPT,
    base_entity_info= context_ent_info,
    agg_entity_info= context_agg_ent_info,
    reasoning_path_info= context_rpath_info,
    relevant_chunk_texts = top_chunks_text
)

print(response)

The authors of the study made a specific architectural modification to the Gated Recurrent Unit (GRU) model from BotScreen to better suit their aimbot detection framework. The original GRU model from BotScreen utilized a mean squared error loss metric. However, the authors modified this model by switching the loss metric to binary cross-entropy loss. This change was made to align with their requirement for binary outputs from the model, which would indicate whether the input data corresponds to an aimbot or not. 

In addition to changing the loss metric, the authors also added a sigmoid activation function to the output of the model. This modification was necessary to bind the output to a range between 0 and 1, effectively providing a probability score that the input data represents an aimbot. 

For the GRU-based model, the authors retained most of the hyper-parameters detailed in the original BotScreen study. These included a 3-layer bidirectional GRU with 10% dropout, 64 hidden units